# hetero-serve — verify the paged-attention CUDA kernels

Runs on a **free Colab T4**. No account setup, no credit card.

**First: `Runtime → Change runtime type → T4 GPU`, then `Run all`.**

This compiles and checks two hand-written CUDA kernels, then measures how close
they get to the card's peak memory bandwidth.

| | |
|---|---|
| **v1** | naive fused kernel — scores in shared memory, scalar loads, shared-memory tree reduction |
| **v2** | online softmax (FlashAttention-style running max/sum, no score vector at all), one warp per (sequence, head), coalesced per-lane slices, `__shfl_down_sync` reductions |

Decode attention is **memory-bandwidth bound** — it reads every cached K and V
exactly once and does almost no arithmetic per byte. So the score that matters is
achieved GB/s against peak, not a speedup over an arbitrary baseline.

Repo: https://github.com/mneha05/hetero-serve

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    bw = p.memory_clock_rate * 1e3 * 2 * (p.memory_bus_width / 8) / 1e9
    print(f'{p.name}: {p.total_memory/1e9:.1f} GB, {p.multi_processor_count} SMs, '
          f'sm_{p.major}{p.minor}, ~{bw:.0f} GB/s peak')
else:
    print('NO GPU -- set Runtime > Change runtime type > T4 GPU, then Run all again.')

In [ ]:
# %cd first so re-running this cell cannot nest hetero-serve/hetero-serve
%cd /content
!rm -rf hetero-serve
!git clone -q https://github.com/mneha05/hetero-serve.git
%cd hetero-serve
# ninja is what torch.utils.cpp_extension shells out to in order to build the
# CUDA extensions; stock Colab does not ship it. apt's build is the reliable
# one -- pip's wheel sometimes lands where the build subprocess cannot see it.
!pip install -q pytest
!apt-get -qq install -y ninja-build 2>/dev/null
!ninja --version && nvcc --version | tail -2


## 1. Correctness

Compiles both kernels and checks them against a torch reference and against each
other. The first run takes ~1–2 minutes because `nvcc` builds the extensions;
they are cached afterwards.

All 34 tests should pass — nothing should skip, because there is a GPU now.

In [ ]:
!HETEROSERVE_VERBOSE_BUILD=1 python -m pytest tests/test_torch_engine.py -v 2>&1 | tail -40

## 2. Bandwidth roofline

Three paths on identical data: host gather, v1, v2. Correctness is verified
before anything is timed — a path that disagrees is printed as FAILED rather
than benchmarked.

In [ ]:
!python scripts/bench_kernel.py --batch 16 --context 512 --dtype float16


In [ ]:
# Longer context. v1 keeps the whole score vector in shared memory so it is
# the one that should struggle; v2 and v3 are O(1) in context length.
!python scripts/bench_kernel.py --batch 32 --context 2048 --dtype float16


## 3. The whole serving system, on GPU

Two workers on the one card, real processes over real sockets, the full
migrate-vs-recompute sweep. `--devices cuda:0,cuda:0` puts both workers on the
same GPU, which is fine for exercising the scheduler (though the interconnect
between them is then a formality — the shaper is what makes the link budget
meaningful).

In [ ]:
!python run_demo.py --devices cuda:0,cuda:0 2>&1 | tail -45

In [ ]:
!python -m heteroserve.bench.sweep --devices cuda:0,cuda:0 --requests 32 --repeats 2 \
    --prefix-tokens 256 --gen 16 --bandwidths 50,200,1000,10000 --num-blocks 384

## 3b. How many context splits?

v3 picks `num_splits` from the device SM count. This sweeps it by hand so you
can see the occupancy effect directly — and where splitting starts costing
more in merge overhead than it wins in parallelism.


In [ ]:
for s in [1, 2, 4, 8, 16, 32]:
    print(f'--- {s} splits ---')
    !python scripts/bench_kernel.py --batch 16 --context 512 --dtype float16 --splits {s} --iters 30 2>&1 | grep -E 'v3 kernel|v2 kernel'


## 4. Nsight Compute (optional, deeper)

If `ncu` is present, this dumps achieved occupancy, memory throughput and warp
stall reasons for the v2 kernel — the numbers you would actually use to decide
what to optimise next.

In [ ]:
# Nsight said v2 was occupancy-starved (0.1 waves). Profile v3's split kernel
# and check whether the context split actually filled the device.
!which ncu && ncu --set full --kernel-name regex:paged_attention_split \n    --launch-count 1 python scripts/bench_kernel.py --batch 16 --context 512 --iters 2 \n    2>&1 | head -60 || echo 'ncu not available in this runtime'
